# 第二课：指标、Mask 与无需训练的预测基线

第一课解决了“一个训练样本是什么”。这一课解决：

1. MAE、RMSE、MAPE 分别在惩罚什么？
2. `mask=False` 为什么必须让梯度为 0？
3. 预测结果为什么必须还原到真实速度单位后再报告？
4. 一个复杂模型至少应该战胜哪些简单方法？
5. 15、30、60 分钟指标是怎样从 `[B,12,N]` 中取出的？

> 这一课仍然不训练神经网络。请看到“先手算”时先写答案，再运行代码。

## 0. 加载项目与 METR-LA

从项目根目录或 `notebooks/` 目录启动 Jupyter。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.evaluation.metrics import masked_metrics, horizon_metrics  # noqa: E402
from crosscity.models import HistoricalAverage  # noqa: E402

config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
raw = bundle.raw_values
raw_mask = bundle.raw_mask
train_end, val_end = bundle.split_points
print('raw:', raw.shape, 'test windows:', len(bundle.test))

## 1. 用三个预测亲手计算 MAE

假设某传感器未来三个时刻的真实速度和预测速度为：

$$y=[50,40,30],\qquad \hat y=[46,46,33]$$

绝对误差为：

$$|\hat y-y|=[4,6,3]$$

**先手算：** MAE 是多少？

In [ ]:
y = torch.tensor([50.0, 40.0, 30.0])
prediction = torch.tensor([46.0, 46.0, 33.0])
absolute_error = (prediction - y).abs()
mae = absolute_error.mean()
print('绝对误差:', absolute_error.tolist())
print('MAE:', mae.item())

$$\mathrm{MAE}=\frac{1}{K}\sum_i|\hat y_i-y_i|$$

MAE 与原始目标单位相同。如果速度单位是 mph，MAE 也是 mph。MAE=4.33 表示每个有效位置平均偏差约 4.33 mph，但不能理解为每次都恰好错 4.33。

## 2. RMSE 为什么更在意大错误

$$\mathrm{RMSE}=\sqrt{\frac{1}{K}\sum_i(\hat y_i-y_i)^2}$$

**先手算：** 上面三个误差的 RMSE 是多少？它应该大于、等于还是小于 MAE？

In [ ]:
rmse = ((prediction - y).square().mean()).sqrt()
print('MAE :', mae.item())
print('RMSE:', rmse.item())

In [ ]:
errors_a = torch.tensor([4.0, 4.0, 4.0, 4.0])
errors_b = torch.tensor([0.0, 0.0, 0.0, 16.0])
for name, errors in [('均匀错误', errors_a), ('一个大错误', errors_b)]:
    print(name, 'MAE=', errors.abs().mean().item(),
          'RMSE=', errors.square().mean().sqrt().item())

两组误差的 MAE 相同，但第二组 RMSE 更大。因此：

- MAE 描述典型绝对偏差，容易直接解释；
- RMSE 更强烈地惩罚少量严重错误；
- 二者一起报告，可以发现模型是否偶尔出现灾难性预测。

## 3. MAPE 是相对误差，但低速时很敏感

$$\mathrm{MAPE}=\frac{100\%}{K}\sum_i\frac{|\hat y_i-y_i|}{|y_i|}$$

**先手算：** 同样是预测偏差 5 mph，当真实速度为 50 mph 和 5 mph 时，相对误差分别是多少？

In [ ]:
for target in (50.0, 5.0, 0.5):
    error = 5.0
    print(f'真实速度={target:4.1f}, 绝对误差=5.0, 百分比误差={error/target*100:7.2f}%')

真实值越接近 0，MAPE 越容易爆炸。项目只在 `mask=True` 且真实速度大于 0 的位置计算，但非常低的有效速度仍会产生很大的百分比误差。因此不要只看 MAPE，也不要把 MAPE 当作训练损失。

## 4. Mask 如何改变指标

现在假设第二个目标没有真实观测。张量为了保持长度 3，在这个位置放了占位值 55：

$$y=[50,\color{gray}{55},30],\qquad m=[1,0,1]$$

预测为 `[46, 10, 33]`。

**先手算：** 如果错误地包含占位值，MAE 是多少？正确排除后又是多少？

In [ ]:
target_with_placeholder = torch.tensor([50.0, 55.0, 30.0])
prediction = torch.tensor([46.0, 10.0, 33.0])
mask = torch.tensor([True, False, True])

wrong_mae = (prediction - target_with_placeholder).abs().mean()
correct_mae = (prediction[mask] - target_with_placeholder[mask]).abs().mean()
print('错误地包含占位值:', wrong_mae.item())
print('只计算真实观测  :', correct_mae.item())

正确公式为：

$$\mathrm{MaskedMAE}=\frac{\sum_i m_i|\hat y_i-y_i|}{\sum_i m_i}$$

分母是有效观测数量，而不是张量总长度。

In [ ]:
metrics = masked_metrics(prediction, target_with_placeholder, mask)
metrics

## 5. Mask 不仅影响指标，还影响反向传播

训练时，损失函数对预测值的导数决定模型参数更新方向。缺失目标位置的梯度必须为 0。下面直接观察三个预测位置的梯度。

**先预测：** 使用 mask 后，三个位置的 `prediction.grad` 分别会是什么特征？

In [ ]:
prediction_for_grad = torch.tensor([46.0, 10.0, 33.0], requires_grad=True)
loss = (prediction_for_grad[mask] - target_with_placeholder[mask]).abs().mean()
loss.backward()
print('prediction.grad:', prediction_for_grad.grad.tolist())
print('中间缺失位置梯度:', prediction_for_grad.grad[1].item())
assert prediction_for_grad.grad[1] == 0


这正是你上一课总结的内容：`mask=False` 的目标没有事实依据，所以它既不应影响报告指标，也不应推动参数更新。

## 6. 为什么指标要在原始速度单位中计算

模型训练时预测的是标准化后的 $z$。如果直接报告标准化空间的 MAE，例如 0.2，它没有直观的交通含义，而且不同城市的标准差不同，无法公平比较。评估前要执行：

$$\hat x=\hat z\sigma_{train}+\mu_{train}$$

In [ ]:
scaled_prediction = torch.tensor([0.0, 0.2, -0.5])
restored_prediction = bundle.scaler.inverse_transform(scaled_prediction)
print('标准化预测:', scaled_prediction.tolist())
print('还原后速度:', restored_prediction.tolist())
print('训练集 mean/std:', bundle.scaler.mean, bundle.scaler.std)

训练损失可以在标准化空间计算，因为它数值稳定；最终 MAE/RMSE/MAPE 必须在逆标准化后计算。

## 7. 基线一：Persistence（保持最后速度）

最简单的交通预测是假设未来不变：

$$\hat X_{t+h,n}=X_{t-1,n},\qquad h=1,\ldots,12$$

它不需要训练，短期内通常很强。复杂模型如果不能战胜它，可能没有学到有用规律。缺失的最后输入在标准化空间被填为 0，逆标准化后对应训练集均值。

In [ ]:
test_starts = bundle.test.indices
output_steps = config.dataset.output_steps

# 从 TrafficDataset 取输入，确保缺失值处理与模型实际看到的完全一致。
last_inputs_scaled = torch.stack([bundle.test[i][0][-1, :, 0] for i in range(len(bundle.test))])
last_inputs = bundle.scaler.inverse_transform(last_inputs_scaled)
persistence_prediction = last_inputs[:, None, :].repeat(1, output_steps, 1)

test_target = torch.from_numpy(np.stack([
    raw[t:t + output_steps] for t in test_starts
])).float()
test_mask = torch.from_numpy(np.stack([
    raw_mask[t:t + output_steps] for t in test_starts
]))

print('prediction:', tuple(persistence_prediction.shape))
print('target    :', tuple(test_target.shape))
print('mask      :', tuple(test_mask.shape))

In [ ]:
persistence_metrics = horizon_metrics(persistence_prediction, test_target, test_mask)
pd.DataFrame(persistence_metrics).T

观察 15、30、60 分钟 MAE。Persistence 通常随着预测距离增加迅速变差，因为“当前速度保持不变”的假设越来越不可信。

## 8. 基线二：Historical Average

Historical Average（HA）利用周周期：对每个传感器、每个星期内的五分钟槽，计算训练期平均速度。

例如，它会用训练期其他星期一 8:00 的观测，预测测试期星期一 8:00。若某个槽没有观测，则退回该传感器的训练期平均速度。

它不查看最近一小时，因此不会响应突发事故，但能表达规律性的通勤模式。

In [ ]:
ha = HistoricalAverage(config.dataset.steps_per_day)
ha.fit(raw[:train_end], raw_mask[:train_end])
ha_prediction = torch.from_numpy(ha.predict(test_starts, output_steps)).float()
ha_metrics = horizon_metrics(ha_prediction, test_target, test_mask)
pd.DataFrame(ha_metrics).T

**先解释再继续：** HA 在不同 horizon 上的误差为什么可能变化得没有 Persistence 那么剧烈？它根本没有使用预测起点之前的即时速度。

## 9. 正确比较两个基线

把结果放在同一张表。MAE/RMSE/MAPE 都是越低越好。

In [ ]:
rows = []
for model_name, result in [('Persistence', persistence_metrics), ('Historical Average', ha_metrics)]:
    for horizon, values in result.items():
        rows.append({'model': model_name, 'horizon': horizon, **values})
comparison = pd.DataFrame(rows)
comparison.pivot(index='horizon', columns='model', values=['mae', 'rmse', 'mape']).round(3)

In [ ]:
order = ['horizon_3', 'horizon_6', 'horizon_12']
labels = ['15 min', '30 min', '60 min']
plt.figure(figsize=(7, 4))
for model_name, result in [('Persistence', persistence_metrics), ('Historical Average', ha_metrics)]:
    plt.plot(labels, [result[h]['mae'] for h in order], marker='o', label=model_name)
plt.ylabel('MAE (speed unit)')
plt.xlabel('Forecast horizon')
plt.title('Simple baselines on METR-LA test set')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

这里不要简单地问“哪个模型最好”，而要问：

- 哪个基线在 15 分钟更强？
- 哪个基线随 horizon 增长更稳定？
- RMSE 与 MAE 的差距是否很大？这是否暗示少数严重错误？
- MAPE 排名是否与 MAE 一致？

## 10. `horizon_3` 到底是什么意思

模型输出 `[B,12,N]`。因为每步 5 分钟：

- `prediction[:, 2, :]` 是第 3 步，即 15 分钟；
- `prediction[:, 5, :]` 是第 6 步，即 30 分钟；
- `prediction[:, 11, :]` 是第 12 步，即 60 分钟。

注意 Python 从 0 开始计数，所以第 3 步索引是 2。`overall` 则把 12 个 horizon 的所有有效位置放在一起计算。

In [ ]:
manual_h3 = masked_metrics(
    persistence_prediction[:, 2, :],
    test_target[:, 2, :],
    test_mask[:, 2, :],
)
print('手工选择第 3 步:', manual_h3)
print('horizon_metrics  :', persistence_metrics['horizon_3'])
assert np.isclose(manual_h3['mae'], persistence_metrics['horizon_3']['mae'])

## 11. 将你的 STGCN 结果放进同一语境

你之前的 METR-LA STGCN 结果是：overall MAE 3.887，15 分钟 3.286，30 分钟 3.857，60 分钟 4.786。现在将它与两个无需训练的基线比较。

In [ ]:
stgcn_from_previous_run = {
    'overall': {'mae': 3.8867681, 'rmse': 7.4323559, 'mape': 11.3166656},
    'horizon_3': {'mae': 3.2856343, 'rmse': 6.0175390, 'mape': 8.9944229},
    'horizon_6': {'mae': 3.8569572, 'rmse': 7.3356605, 'mape': 11.1707792},
    'horizon_12': {'mae': 4.7862101, 'rmse': 9.1473846, 'mape': 14.8256321},
}

all_rows = []
for name, result in [
    ('Persistence', persistence_metrics),
    ('Historical Average', ha_metrics),
    ('STGCN (previous run)', stgcn_from_previous_run),
]:
    for horizon, values in result.items():
        all_rows.append({'model': name, 'horizon': horizon, **values})
all_results = pd.DataFrame(all_rows)
all_results.pivot(index='horizon', columns='model', values='mae').round(3)

**分析任务：**

1. STGCN 是否在每个 horizon 都战胜 Persistence？
2. 如果没有，这是否意味着 STGCN 完全无效，还是意味着当前实现/训练仍有改进空间？
3. STGCN 的 RMSE 明显大于 MAE 吗？这说明什么？
4. 不要拿本 notebook 的结果与不同 split、不同 mask 规则的论文数字直接比较。

## 12. 本课验收

请不用查看前文回答：

1. MAE=3 mph 的自然语言含义是什么？
2. 两个模型 MAE 相同但 RMSE 不同，RMSE 更大的模型发生了什么？
3. 为什么真实速度接近 0 时 MAPE 不稳定？
4. `mask=False` 的位置为什么不仅不能参与指标，也不能参与反向传播？
5. 为什么不能直接报告标准化空间中的 MAE？
6. Persistence 和 Historical Average 分别依赖哪一种假设？
7. `horizon_6` 是未来多少分钟，代码索引是多少？

运行指标测试：

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_metrics.py'],
    cwd=ROOT, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, '指标测试未通过，请把错误发给我'

## 13. 请带回来的结果

运行后请记录：

- Persistence 的 overall、15、30、60 分钟 MAE；
- Historical Average 的对应 MAE；
- 三个模型中各 horizon 的最佳者；
- STGCN 是否稳定战胜简单基线；
- 七个验收问题中最不确定的一项。

如果 STGCN 没有战胜某个简单基线，不要修改或隐藏结果。下一课学习共享 MLP 与训练循环时，我们会用它诊断数据管线、优化过程和模型复杂度。